In [17]:
# load the training data
import re
import pandas as pd
import numpy as np
import torch

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print("train shape:", train.shape)
train.head()

train shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [18]:
# get topic words from the prompts
topics = set()
for prompt in train["prompt"]:
    words = re.findall(r"[A-Z][a-z]+", prompt)
    for w in words:
        if len(w) > 3 and w not in ["Pick", "Select", "Choose", "Which", "What", "Determine", "Identify"]:
            topics.add(w)

topics = list(topics)[:50]
print(f"got {len(topics)} topics")
print("first 10:", topics[:10])

got 50 topics
first 10: ['Bruno', 'Acoustic', 'Townsend', 'Maxwell', 'Doppler', 'Ratio', 'Fermi', 'Hamiltonians', 'Young', 'Wilson']


In [19]:
# fetch wikipedia articles using the rest api
import requests

articles = []
for topic in topics:
    slug = topic.replace(" ", "_")
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{slug}"
    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "student-project/1.0"})
        if r.status_code == 200:
            data = r.json()
            text = data.get("extract", "").strip()
            if len(text) > 100:
                articles.append({"topic": topic, "text": text})
    except:
        continue

print(f"fetched {len(articles)} articles")
print()
print("example article:")
print(articles[0]["text"][:300])

fetched 15 articles

example article:
The Doppler effect is the change in the frequency or, equivalently, the period of a wave in relation to an observer who is moving relative to the source of the wave. It is named after the physicist Christian Doppler, who described the phenomenon in 1842. A common example of Doppler shift is the chan


In [20]:
# split articles into sentences
sentences = []
for article in articles:
    for s in article["text"].split(". "):
        s = s.strip()
        if len(s) > 20:
            sentences.append(s)

print(f"total sentences: {len(sentences)}")
print("example:", sentences[0])

total sentences: 45
example: The Doppler effect is the change in the frequency or, equivalently, the period of a wave in relation to an observer who is moving relative to the source of the wave


In [21]:
# load a sentence embedding model using transformers
from transformers import AutoTokenizer, AutoModel

emb_model_name = "sentence-transformers/all-MiniLM-L6-v2"
emb_tokenizer = AutoTokenizer.from_pretrained(emb_model_name)
emb_model = AutoModel.from_pretrained(emb_model_name).to("cuda")
emb_model.eval()

def embed_texts(texts, batch_size=32):
    all_embeds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = emb_tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = emb_model(**enc)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeds.append(pooled.cpu())
    return torch.cat(all_embeds, dim=0)

print("embedder loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedder loaded


In [22]:
# embed all sentences
sentence_embeddings = embed_texts(sentences)
print("shape:", sentence_embeddings.shape)

shape: torch.Size([45, 384])


In [23]:
# retrieval function using cosine similarity
def retrieve(query, k=3):
    query_emb = embed_texts([query])
    scores = (sentence_embeddings @ query_emb.T).squeeze(1)
    top_k = torch.topk(scores, k=k)
    return [sentences[i] for i in top_k.indices]

test = retrieve("Who was Isaac Newton?", k=3)
for s in test:
    print("-", s)

- He was one of very few physicists to excel in both theoretical and experimental physics
- He made significant contributions to the development of statistical mechanics, quantum theory, and nuclear and particle physics.
- Townes and Arthur Leonard Schawlow and the optical amplifier patented by Gordon Gould.


In [24]:
# load the language model
from transformers import AutoModelForCausalLM

llm_name = "Qwen/Qwen2.5-1.5B-Instruct"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16,
    device_map="cuda",
)
llm_model.eval()
print("llm loaded")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

llm loaded


In [25]:
# function to answer a question using retrieved context
def answer_question(question, options):
    context = retrieve(question, k=3)
    context_str = "\n".join(context)

    prompt = f"""Use the context below to answer the multiple choice question.

Context:
{context_str}

Question: {question}

Options:
A. {options['A']}
B. {options['B']}
C. {options['C']}
D. {options['D']}
E. {options['E']}

Answer with only the letter (A, B, C, D, or E):"""

    inputs = llm_tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = llm_model.generate(**inputs, max_new_tokens=3, do_sample=False, pad_token_id=llm_tokenizer.eos_token_id)
    generated = out[0][inputs["input_ids"].shape[1]:]
    return llm_tokenizer.decode(generated, skip_special_tokens=True).strip()

sample_row = train.iloc[0]
sample_options = {L: sample_row[L] for L in "ABCDE"}
print("example question:", sample_row["prompt"][:100])
print("actual answer:", sample_row["answer"])
print("model output:", answer_question(sample_row["prompt"], sample_options))

example question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and 
actual answer: B
model output: A

The


In [26]:
# run rag on the test set for submission
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

predictions = []
for i in range(len(test)):
    row = test.iloc[i]
    options = {L: row[L] for L in "ABCDE"}
    pred = answer_question(row["prompt"], options)
    predictions.append(pred)
    if (i+1) % 20 == 0:
        print(f"done {i+1} / {len(test)}")

test["prediction"] = predictions
print()
print("all done")

done 20 / 500
done 40 / 500
done 60 / 500
done 80 / 500
done 100 / 500
done 120 / 500
done 140 / 500
done 160 / 500
done 180 / 500
done 200 / 500
done 220 / 500
done 240 / 500
done 260 / 500
done 280 / 500
done 300 / 500
done 320 / 500
done 340 / 500
done 360 / 500
done 380 / 500
done 400 / 500
done 420 / 500
done 440 / 500
done 460 / 500
done 480 / 500
done 500 / 500

all done


In [27]:
# build top-3 predictions for submission format
def extract_letter(text):
    m = re.search(r"[A-E]", str(text))
    return m.group(0) if m else "A"

top3_preds = []
common_letters = ["B", "C", "A", "E", "D"]
for i in range(len(test)):
    pred = extract_letter(test.iloc[i]["prediction"])
    top3 = [pred]
    for L in common_letters:
        if L not in top3:
            top3.append(L)
        if len(top3) == 3:
            break
    top3_preds.append(" ".join(top3))

test["top3"] = top3_preds
print(test[["id", "top3"]].head())

   id   top3
0   1  A B C
1   2  B C A
2   3  E B C
3   4  A B C
4   5  A B C


In [28]:
# save submission file
submission = test[["id", "top3"]].rename(columns={"top3": "Prediction"})
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("saved to /kaggle/working/submission.csv")
print("rows:", len(submission))
print(submission.head())

saved to /kaggle/working/submission.csv
rows: 500
   id Prediction
0   1      A B C
1   2      B C A
2   3      E B C
3   4      A B C
4   5      A B C
